In [1]:
import sys
sys.path.append("../src")

import fred_api
import analysis
import charts
import db
import summaries
import user_intent
import date_parser
import time

In [2]:
user_question="Compare yearly unemployment averages across 2020 and 2022" #CPIAUCSL	

# # STEP 1 determine question intent via logreg
# start = time.perf_counter()
# question_intent=user_intent.predict_intent(user_question)
# print(f"Step 1 took: {time.perf_counter() - start:.4f}s")
# print(f"inferred user questions intent: {question_intent}")

# # STEP 2 determine what dataset to use via fuzzywuzzy
# start = time.perf_counter()
# metadata=db.get_series_metadata()
# series_ids=user_intent.predict_series_intent(user_question,metadata)
# print(f"Step 2 took: {time.perf_counter() - start:.4f}s")
# print(f"inferred dataset intent: {series_ids}")

# # STEP 3 determine the date aggregation grain
# start = time.perf_counter()
# date_grain=user_intent.date_aggregation_grain_intent(user_question)
# print(f"Step 3 took: {time.perf_counter() - start:.4f}s")
# print(f"inferred date aggregation grain: {date_grain}")

# # STEP 4 determine time frame of dataset via claude LLM & validate output
# start = time.perf_counter()
# start_date, end_date=user_intent.timeline_intent(user_question,date_grain)
# print(f"Step 4 took: {time.perf_counter() - start:.4f}s")
# print(f"inferred dataset start and end date: {start_date},{end_date}")

In [3]:
# DATA_PLANNER={
# "question_intent":question_intent,
# "series_ids":series_ids,
# "date_grain":date_grain,
# "start_date":start_date,
# "end_date":end_date
# }

# DATA_PLANNER

In [4]:
# # ---------------------DATA PREPERATION-----------------
# # STEP 0 pull the series metadata
# dataset_context=summaries.build_context(series_ids)

# # STEP 1 pull relevant data (does the limiting and pulls date grain needed for aggergation)
# start = time.perf_counter()
# series_sql = ",".join(f"'{x}'" for x in DATA_PLANNER["series_ids"])
# query=f"""SELECT *,EXTRACT({DATA_PLANNER["date_grain"]} from date) as '{DATA_PLANNER["date_grain"]}' 
# FROM clean_fred_data 
# WHERE series_id in ({series_sql}) 
# and date between '{DATA_PLANNER["start_date"]}' and '{DATA_PLANNER["end_date"]}'"""
# df=db.run_query(query)
# df

In [2]:
import data_planner
df,dataset_context,DATA_PLANNER=data_planner.build_data_plan("Rank GDP quarters from highest to lowest.")
df

,date,value,series_id,QUARTER
0,2016-01-01,18525.933,GDP,1
1,2016-04-01,18711.702,GDP,2
2,2016-07-01,18892.639,GDP,3
3,2016-10-01,19089.379,GDP,4
4,2017-01-01,19280.084,GDP,1
5,2017-04-01,19438.643,GDP,2
6,2017-07-01,19692.595,GDP,3
7,2017-10-01,20037.088,GDP,4
8,2018-01-01,20328.553,GDP,1
9,2018-04-01,20580.912,GDP,2


In [7]:
import semantics
series_semantics=semantics.dataset_ranking_semantics(DATA_PLANNER['series_ids'])
prompt=summaries.build_ranking_method_prompt("Rank GDP quarters from highest to lowest.",series_semantics)
result=summaries.run_prompt(prompt)

ascending_bool=result.split(",")[0] == "True"
n=int(result.split(",")[1])

df_agg=analysis.compute_df_aggregation(df,group_by_fields=DATA_PLANNER['date_grain'],computation="mean")
sorted_df=analysis.rank_periods(df_agg, sort_by="calculated_value", n=n, ascending=ascending_bool)


In [8]:
sorted_df

,QUARTER,calculated_value
3,4,24532.1255
2,3,24180.8792
1,2,23653.7516
0,1,23489.5167


False

In [ ]:
import comparison
comparison.run_comparison_analysis(df,user_question,DATA_PLANNER,dataset_context)